# Paper 2 final evidence: ECE diagnostic supplement
This notebook summarizes the WA-calibrated Guarded-on-ECE evidence and exports diagnostic tables and figures. ECE observations are evaluation-only.


## 1. Locate the evidence bundle
The next cell resolves the repository and canonical ECE v3 split for reproducible outputs.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    parent for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (parent / "data" / "splits").is_dir() and (parent / "notebooks").is_dir()
)
EVIDENCE_DIR = PROJECT_ROOT / "notebooks/experiment/paper2-final-evidence-1.0"
ECE_DIR = EVIDENCE_DIR / "ece_guarded"
FIGURES_DIR = EVIDENCE_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Evidence directory: {EVIDENCE_DIR}")

Evidence directory: /scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/paper2-final-evidence-1.0


## 2. Check the unseen ECE split and site descriptors
The canonical split has five unseen stations and 30 evaluation rows per station; site descriptors support the lowland deployment context.

In [2]:
ECE_SPLIT = PROJECT_ROOT / "data/splits/derived_8.4_ece_v3"
ece_test = pd.read_csv(ECE_SPLIT / "test.csv", low_memory=False)
site_features = pd.read_csv(ECE_SPLIT / "station_static_features.csv")
assert len(ece_test) == 150
assert ece_test["station_id"].nunique() == 5
assert bool((ece_test["station_id"].value_counts() == 30).all())
site_columns = ["station_id", "J_elev_m", "J_bio_bio12"]
site_descriptors = site_features[site_columns].sort_values("station_id")
print(site_descriptors.to_string(index=False))

             station_id  J_elev_m  J_bio_bio12
    ECE_BBG_Lost_Meadow        52         1019
        ECE_BBG_Main_St        51         1018
ECE_Renton_Garden_North       157         1227
 ECE_Renton_Garden_Shed       157         1227
        ECE_Renton_Home       136         1181


## 3. Validate the complete ECE run and export T7/T9 tables
The next cell checks seed, family, and policy coverage; verifies the WA-only label map; and writes pooled and per-station policy tables.

In [3]:
summary = pd.read_csv(ECE_DIR / "summary.csv")
seed_metrics = pd.read_csv(ECE_DIR / "seed_metrics.csv")
station_metrics = pd.read_csv(ECE_DIR / "station_metrics.csv")
predictions = pd.read_csv(ECE_DIR / "predictions_v3.csv")
wa_calibration = pd.read_csv(ECE_DIR / "wa_calibration.csv")
with (ECE_DIR / "routing_audit.json").open(encoding="utf-8") as handle:
    routing_audit = json.load(handle)

wa_dir = PROJECT_ROOT / "data/splits/derived_8.4"
wa_stations = set(pd.read_csv(wa_dir / "train.csv", usecols=["station_id"])["station_id"])
wa_stations.update(pd.read_csv(wa_dir / "val.csv", usecols=["station_id"])["station_id"])
assert wa_stations.isdisjoint(set(ece_test["station_id"]))

families = ["Clustering_V0_Full_k2", "Clustering_Backbone54_k2", "Guarded_Backbone54_k2"]
policies = ["as_routed", "auto_hard", "auto_soft", "c0_only", "c1_only"]
seeds = [42, 7, 13, 101, 123]
assert set(seed_metrics["family"]) == set(families + ["Global_Single_54"])
assert set(seed_metrics["policy"]) == set(policies + ["direct"])
assert len(seed_metrics) == 80
assert len(station_metrics) == 400
assert len(predictions) == 11250
assert bool((summary.loc[summary["policy"].isin(["c0_only", "c1_only"]), "deployable"] == False).all())
assert routing_audit["ece_rows_used_for_fitting"] is False
expert_map = routing_audit["semantic_expert_indices"]
assert expert_map["Clustering_V0_Full_k2"]["dry_expert_index"] == 0
assert expert_map["Clustering_V0_Full_k2"]["wet_expert_index"] == 1
assert expert_map["Clustering_Backbone54_k2"]["dry_expert_index"] == 0
assert expert_map["Clustering_Backbone54_k2"]["wet_expert_index"] == 1
assert expert_map["Guarded_Backbone54_k2"]["dry_expert_index"] == 1
assert expert_map["Guarded_Backbone54_k2"]["wet_expert_index"] == 0
assert expert_map["Clustering_V0_Full_k2"]["semantic_dry_matches_canonical_feature_minimum"] is False
assert expert_map["Clustering_Backbone54_k2"]["semantic_dry_matches_canonical_feature_minimum"] is False
assert expert_map["Guarded_Backbone54_k2"]["semantic_dry_matches_canonical_feature_minimum"] is True
for family in families:
    for policy in policies:
        rows = seed_metrics.query("family == @family and policy == @policy")
        assert sorted(rows["seed"].astype(int).tolist()) == sorted(seeds)

site_descriptors.to_csv(ECE_DIR / "site_descriptors.csv", index=False)
summary.to_csv(ECE_DIR / "ece_policy_summary.csv", index=False)
station_summary = station_metrics.groupby(["family", "policy", "station"], as_index=False).agg(
    rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"),
    bias_mean=("bias", "mean"), ubrmse_mean=("ubrmse", "mean"))
station_summary.to_csv(ECE_DIR / "ece_station_policy_summary.csv", index=False)
print(f"WA stations={len(wa_stations)}; ECE stations={ece_test['station_id'].nunique()} (disjoint)")
print(f"seed_metrics rows={len(seed_metrics)} station_metrics rows={len(station_metrics)} predictions rows={len(predictions)}")
print(summary[["family", "policy", "deployable", "rmse_mean", "rmse_std", "bias_mean", "ubrmse_mean"]].to_string(index=False))
print("WA expert mapping (salvage c0 preserved for V0/Backbone; Guarded canonical c1 is dry):")
print(json.dumps(expert_map, indent=2))


WA stations=7; ECE stations=5 (disjoint)
seed_metrics rows=80 station_metrics rows=400 predictions rows=11250
                  family    policy  deployable  rmse_mean  rmse_std  bias_mean  ubrmse_mean
   Clustering_V0_Full_k2 as_routed        True   0.165908  0.003051   0.117727     0.116899
   Clustering_V0_Full_k2 auto_hard        True   0.057768  0.000617   0.028654     0.050158
   Clustering_V0_Full_k2 auto_soft        True   0.057768  0.000617   0.028655     0.050158
   Clustering_V0_Full_k2   c0_only       False   0.057768  0.000617   0.028654     0.050158
   Clustering_V0_Full_k2   c1_only       False   0.192536  0.004021   0.185650     0.051020
Clustering_Backbone54_k2 as_routed        True   0.167431  0.003401   0.141932     0.088816
Clustering_Backbone54_k2 auto_hard        True   0.057768  0.000617   0.028654     0.050158
Clustering_Backbone54_k2 auto_soft        True   0.057768  0.000617   0.028655     0.050158
Clustering_Backbone54_k2   c0_only       False   0.057768  0.0

### WA synthetic-SMAP-mask calibration check
This cell confirms every router family has WA validation results for the synthetic full-SMAP-missing regime; no ECE rows enter this calibration.

In [4]:
assert set(wa_calibration["family"]) == set(families)
masked_calibration = wa_calibration.query("setting == 'smap_masked_val'")
assert len(masked_calibration) == 6
assert set(masked_calibration["policy"]) == {"static_hard_masked", "aux_hard_masked"}
assert set(masked_calibration["family"]) == set(families)
assert routing_audit["ece_rows_used_for_fitting"] is False
print(masked_calibration.sort_values(["family", "policy"]).to_string(index=False))

                  family         setting             policy     rmse
Clustering_Backbone54_k2 smap_masked_val    aux_hard_masked 0.073965
Clustering_Backbone54_k2 smap_masked_val static_hard_masked 0.086870
   Clustering_V0_Full_k2 smap_masked_val    aux_hard_masked 0.072757
   Clustering_V0_Full_k2 smap_masked_val static_hard_masked 0.068082
   Guarded_Backbone54_k2 smap_masked_val    aux_hard_masked 0.117180
   Guarded_Backbone54_k2 smap_masked_val static_hard_masked 0.086870


## 4. Export semantic routing shares and calibration evidence
The next cell derives dry/wet specialist weights from each row’s recorded local expert map and prints the WA synthetic-mask calibration table.

In [5]:
predictions["dry_weight"] = np.where(
    predictions["dry_expert_index"].astype(int) == 0,
    predictions["w0"], predictions["w1"])
predictions["wet_weight"] = 1.0 - predictions["dry_weight"]
share_keys = ["family", "policy"]
routing_shares = predictions.groupby(share_keys, as_index=False).agg(
    dry_weight_mean=("dry_weight", "mean"), wet_weight_mean=("wet_weight", "mean"),
    dry_dominant_fraction=("dry_weight", lambda values: float((values > 0.5).mean())))
routing_shares.to_csv(ECE_DIR / "ece_routing_shares.csv", index=False)
station_shares = predictions.groupby(["family", "policy", "station_id"], as_index=False).agg(
    dry_weight_mean=("dry_weight", "mean"), wet_weight_mean=("wet_weight", "mean"))
station_shares.to_csv(ECE_DIR / "ece_routing_shares_by_station.csv", index=False)
print(routing_shares.to_string(index=False))
print("WA-only calibration rows:")
print(wa_calibration.to_string(index=False))

                  family    policy  dry_weight_mean  wet_weight_mean  dry_dominant_fraction
Clustering_Backbone54_k2 as_routed         0.306667     6.933333e-01               0.306667
Clustering_Backbone54_k2 auto_hard         1.000000     0.000000e+00               1.000000
Clustering_Backbone54_k2 auto_soft         1.000000     4.564617e-07               1.000000
Clustering_Backbone54_k2   c0_only         1.000000     0.000000e+00               1.000000
Clustering_Backbone54_k2   c1_only         0.000000     1.000000e+00               0.000000
   Clustering_V0_Full_k2 as_routed         0.460000     5.400000e-01               0.460000
   Clustering_V0_Full_k2 auto_hard         1.000000     0.000000e+00               1.000000
   Clustering_V0_Full_k2 auto_soft         0.999999     6.112629e-07               1.000000
   Clustering_V0_Full_k2   c0_only         1.000000     0.000000e+00               1.000000
   Clustering_V0_Full_k2   c1_only         0.000000     1.000000e+00            

## 4a. Export the semantic policy crosswalk
This crosswalk makes explicit how historical c0/c1 policy IDs map to dry/wet expert semantics and family-local indices before any between-experiment comparison.

In [6]:
policy_crosswalk_rows = []
for family, family_map in expert_map.items():
    for policy in policies:
        if policy == "c0_only":
            semantic_label = "manual dry-expert oracle"
            expert_index = int(family_map["dry_expert_index"])
        elif policy == "c1_only":
            semantic_label = "wet-expert diagnostic"
            expert_index = int(family_map["wet_expert_index"])
        else:
            semantic_label = {
                "as_routed": "static router prediction",
                "auto_hard": "WA-calibrated availability-gated prediction",
                "auto_soft": "WA-calibrated availability-gated blend",
            }[policy]
            expert_index = None
        policy_crosswalk_rows.append({
            "family": family, "policy_id": policy,
            "semantic_label": semantic_label,
            "family_local_expert_index": expert_index,
            "dry_expert_index": int(family_map["dry_expert_index"]),
            "wet_expert_index": int(family_map["wet_expert_index"]),
            "mapping_source": family_map["mapping_source"],
            "wa_canonical_feature_drier_index": int(
                family_map["wa_canonical_feature_drier_index"]
            ),
            "semantic_dry_matches_canonical_feature_minimum": bool(
                family_map["semantic_dry_matches_canonical_feature_minimum"]
            ),
            "deployable": policy in {"as_routed", "auto_hard", "auto_soft"},
            "ece_used_for_mapping_or_calibration": False,
        })
policy_crosswalk_rows.append({
    "family": "Global_Single_54", "policy_id": "direct",
    "semantic_label": "direct global reference",
    "family_local_expert_index": None, "dry_expert_index": None,
    "wet_expert_index": None, "mapping_source": "not_applicable",
    "wa_canonical_feature_drier_index": None,
    "semantic_dry_matches_canonical_feature_minimum": None,
    "deployable": True, "ece_used_for_mapping_or_calibration": False,
})
policy_crosswalk = pd.DataFrame(policy_crosswalk_rows)
policy_crosswalk.to_csv(ECE_DIR / "policy_crosswalk.csv", index=False)
print(policy_crosswalk.to_string(index=False))


                  family policy_id                              semantic_label  family_local_expert_index  dry_expert_index  wet_expert_index                                       mapping_source  wa_canonical_feature_drier_index semantic_dry_matches_canonical_feature_minimum  deployable  ece_used_for_mapping_or_calibration
   Clustering_V0_Full_k2 as_routed                    static router prediction                        NaN               0.0               1.0 derived_8.4-ece-router-salvage-2.0 c0=dry convention                               1.0                                          False        True                                False
   Clustering_V0_Full_k2 auto_hard WA-calibrated availability-gated prediction                        NaN               0.0               1.0 derived_8.4-ece-router-salvage-2.0 c0=dry convention                               1.0                                          False        True                                False
   Clustering_V0_Full_k2 

## 5. Plot T7 and T9 ECE diagnostics
This cell plots the exported policy and routing evidence.

In [7]:
ece_policy = pd.read_csv(ECE_DIR / "ece_policy_summary.csv")
ece_station = pd.read_csv(ECE_DIR / "ece_station_policy_summary.csv")
route_station = pd.read_csv(ECE_DIR / "ece_routing_shares_by_station.csv")
site_descriptors = pd.read_csv(ECE_DIR / "site_descriptors.csv")

display_policy = {
    "as_routed": "Static route",
    "auto_hard": "Availability gate",
    "auto_soft": "Gate + blend",
    "c0_only": "Manual dry oracle*",
    "c1_only": "Wet specialist diagnostic*",
    "direct": "Global direct",
}
family_order = [
    "Clustering_V0_Full_k2", "Clustering_Backbone54_k2",
    "Guarded_Backbone54_k2", "Global_Single_54",
]
family_label = {
    "Clustering_V0_Full_k2": "V0",
    "Clustering_Backbone54_k2": "Backbone",
    "Guarded_Backbone54_k2": "Guarded",
    "Global_Single_54": "Global",
}
policy_order = ["as_routed", "auto_hard", "auto_soft", "c0_only", "c1_only", "direct"]
policy_colors = dict(zip(policy_order, plt.cm.tab10.colors[:len(policy_order)]))

fig, axes = plt.subplots(2, 2, figsize=(17, 11))
ax = axes[0, 0]
x = np.arange(len(family_order))
bar_width = 0.12
for j, policy in enumerate(policy_order):
    rows = ece_policy[ece_policy["policy"] == policy].set_index("family")
    positions = x + (j - (len(policy_order) - 1) / 2) * bar_width
    means, errors = [], []
    for family in family_order:
        if family in rows.index:
            means.append(float(rows.loc[family, "rmse_mean"]))
            errors.append(float(rows.loc[family, "rmse_std"]))
        else:
            means.append(np.nan)
            errors.append(0.0)
    ax.bar(positions, means, bar_width, yerr=errors, capsize=2,
           color=policy_colors[policy], label=display_policy[policy])
ax.set_xticks(x, [family_label[f] for f in family_order])
ax.set_ylabel("ECE RMSE (soil-moisture units)")
ax.set_title("T7 · pooled RMSE by policy (mean ± seed SD)")
ax.grid(axis="y", alpha=0.25)
ax.legend(fontsize=8, ncol=2)

ax = axes[0, 1]
guarded_station = ece_station[ece_station["family"] == "Guarded_Backbone54_k2"]
station_order = sorted(guarded_station["station"].unique())
x = np.arange(len(station_order))
bar_width = 0.14
for j, policy in enumerate(policy_order[:-1]):
    rows = guarded_station[guarded_station["policy"] == policy].set_index("station")
    positions = x + (j - 2) * bar_width
    means = [float(rows.loc[s, "rmse_mean"]) for s in station_order]
    errors = [float(rows.loc[s, "rmse_std"]) for s in station_order]
    ax.bar(positions, means, bar_width, yerr=errors, capsize=2,
           color=policy_colors[policy], label=display_policy[policy])
ax.set_xticks(x, [s.replace("ECE_", "").replace("_", " ") for s in station_order], fontsize=8)
ax.set_ylabel("ECE RMSE")
ax.set_title("T7 · Guarded per-station RMSE (mean ± seed SD)")
ax.grid(axis="y", alpha=0.25)
ax.legend(fontsize=8, ncol=2)

ax = axes[1, 0]
guarded_route = route_station[route_station["family"] == "Guarded_Backbone54_k2"]
descriptor_order = site_descriptors.set_index("station_id").loc[station_order]
for policy in ["as_routed", "auto_hard", "auto_soft"]:
    rows = guarded_route[guarded_route["policy"] == policy].set_index("station_id").loc[station_order]
    ax.plot(np.arange(len(station_order)), rows["dry_weight_mean"].to_numpy(),
            marker="o", linewidth=2, color=policy_colors[policy],
            label=display_policy[policy])
ax.set_xticks(np.arange(len(station_order)), [
    f"{s.replace('ECE_', '').replace('_', ' ')} ({int(e)}m)"
    for s, e in zip(station_order, descriptor_order["J_elev_m"])
], fontsize=8)
ax.set_ylim(-0.05, 1.05)
ax.set_ylabel("Mean dry-expert weight")
ax.set_title("T9 · Guarded routing share by site")
ax.grid(axis="y", alpha=0.25)
ax.legend(fontsize=8)

ax = axes[1, 1]
label_offsets = {
    "ECE_BBG_Main_St": (-64, -20),
    "ECE_BBG_Lost_Meadow": (7, 8),
    "ECE_Renton_Garden_North": (-116, 15),
    "ECE_Renton_Garden_Shed": (-114, -24),
    "ECE_Renton_Home": (7, 8),
}
for _, row in site_descriptors.iterrows():
    ax.scatter(row["J_elev_m"], row["J_bio_bio12"], color="#286c8e", s=48)
    short = row["station_id"].replace("ECE_", "").replace("_", " ")
    ax.annotate(short, (row["J_elev_m"], row["J_bio_bio12"]),
                xytext=label_offsets[row["station_id"]], textcoords="offset points",
                fontsize=8)
ax.set_xlabel("Elevation (m)")
ax.set_ylabel("Annual precipitation proxy (J_bio_bio12)")
ax.set_title("T9 · ECE site descriptors")
ax.grid(alpha=0.25)

fig.suptitle("ECE v3 diagnostics · five unseen stations · 30-day window", fontsize=15)
fig.text(0.01, 0.01, "* Manual dry oracle and wet-specialist diagnostic are non-deployable. "
         "Guarded c1 maps to the dry expert; policy weights use the WA-only label audit.",
         fontsize=9)
fig.tight_layout(rect=(0, 0.04, 1, 0.96))
ece_t7_t9_path = FIGURES_DIR / "T7_T9_ece_diagnostics.png"
fig.savefig(ece_t7_t9_path, dpi=200, bbox_inches="tight")
plt.close(fig)

print(ece_policy[["family", "policy", "deployable", "rmse_mean", "rmse_std",
                  "bias_mean", "ubrmse_mean"]].to_string(index=False))
print(f"Saved figure: {ece_t7_t9_path}")


                  family    policy  deployable  rmse_mean  rmse_std  bias_mean  ubrmse_mean
   Clustering_V0_Full_k2 as_routed        True   0.165908  0.003051   0.117727     0.116899
   Clustering_V0_Full_k2 auto_hard        True   0.057768  0.000617   0.028654     0.050158
   Clustering_V0_Full_k2 auto_soft        True   0.057768  0.000617   0.028655     0.050158
   Clustering_V0_Full_k2   c0_only       False   0.057768  0.000617   0.028654     0.050158
   Clustering_V0_Full_k2   c1_only       False   0.192536  0.004021   0.185650     0.051020
Clustering_Backbone54_k2 as_routed        True   0.167431  0.003401   0.141932     0.088816
Clustering_Backbone54_k2 auto_hard        True   0.057768  0.000617   0.028654     0.050158
Clustering_Backbone54_k2 auto_soft        True   0.057768  0.000617   0.028655     0.050158
Clustering_Backbone54_k2   c0_only       False   0.057768  0.000617   0.028654     0.050158
Clustering_Backbone54_k2   c1_only       False   0.192536  0.004021   0.185650  

## 6. Plot the Guarded F5 prediction overlay
This cell averages the five fixed-router expert seeds by date and exports the daily table used by the five-station overlay. The manual dry oracle and wet specialist diagnostic are labeled as non-deployable.

In [8]:
guarded_family = "Guarded_Backbone54_k2"
overlay_policies = ["as_routed", "auto_hard", "c0_only", "c1_only"]
overlay = predictions[
    (predictions["family"] == guarded_family)
    & predictions["policy"].isin(overlay_policies)
].copy()
overlay["date"] = pd.to_datetime(overlay["date"])
assert overlay["seed"].nunique() == 5
assert set(overlay["station_id"]) == set(ece_test["station_id"])
assert set(overlay["policy"]) == set(overlay_policies)
overlay_daily = overlay.groupby(
    ["station_id", "date", "policy"], as_index=False
).agg(y_true=("y_true", "mean"), y_pred_mean=("y_pred", "mean"),
      y_pred_seed_sd=("y_pred", "std"))
target_checks = overlay.groupby(["station_id", "date"])["y_true"].nunique()
assert bool((target_checks == 1).all())
overlay_daily.to_csv(ECE_DIR / "f5_guarded_daily_overlay.csv", index=False)

display_overlay = {
    "as_routed": "Guarded · static route",
    "auto_hard": "Guarded · WA-calibrated availability gate",
    "c0_only": "Manual dry oracle*",
    "c1_only": "Wet specialist diagnostic*",
}
overlay_colors = {
    "as_routed": "#444444", "auto_hard": "#1b9e77",
    "c0_only": "#377eb8", "c1_only": "#d95f02",
}
fig, axes = plt.subplots(5, 1, figsize=(12, 13), sharex=True, sharey=True)
for ax, station in zip(axes, sorted(overlay_daily["station_id"].unique())):
    station_data = overlay_daily[overlay_daily["station_id"] == station]
    observed = station_data.groupby("date", as_index=False)["y_true"].mean()
    ax.plot(observed["date"], observed["y_true"], color="black", linewidth=1.5,
            label="Observed in-situ target")
    for policy in overlay_policies:
        rows = station_data[station_data["policy"] == policy].sort_values("date")
        ax.plot(rows["date"], rows["y_pred_mean"], color=overlay_colors[policy],
                linewidth=1.2, label=display_overlay[policy])
    ax.set_title(station.replace("ECE_", "").replace("_", " "), loc="left", fontsize=10)
    ax.grid(alpha=0.2)
    ax.set_ylabel("Soil moisture")
axes[-1].set_xlabel("Date")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, 0.035))
fig.suptitle("F5 · Guarded ECE v3 prediction overlay (mean across five expert seeds)",
             y=0.995, fontsize=14)
fig.text(0.01, 0.005, "* Manual dry oracle and wet-specialist diagnostic are non-deployable; "
         "daily means span five seeds.",
         fontsize=8)
fig.tight_layout(rect=(0, 0.10, 1, 0.98))
f5_path = FIGURES_DIR / "F5_guarded_ece_overlay.png"
fig.savefig(f5_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Daily table rows={len(overlay_daily)}")
print(f"Saved figure: {f5_path}")


Daily table rows=600
Saved figure: /scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/paper2-final-evidence-1.0/figures/F5_guarded_ece_overlay.png


## 7. Pair the dry-expert results with salvage C0
This comparison preserves the salvage C0=dry convention for V0 and Backbone, then compares Guarded local c1=dry against salvage Backbone C0 by seed. Every oracle row remains non-deployable.

In [9]:
salvage_seed = pd.read_csv(EVIDENCE_DIR / "comparators/ece_router_salvage_2.0_seed_metrics.csv")
comparison_specs = [
    ("V0 current vs salvage C0", "Clustering_V0_Full_k2", "Clustering_V0_Full_k2", 0),
    ("Backbone current vs salvage C0", "Clustering_Backbone54_k2", "Clustering_Backbone54_k2", 0),
    ("Guarded local c1 dry vs salvage Backbone C0", "Guarded_Backbone54_k2", "Clustering_Backbone54_k2", 1),
]
paired_rows = []
for comparison_label, current_family, reference_family, current_dry_index in comparison_specs:
    current = seed_metrics.query(
        "family == @current_family and policy == 'c0_only'"
    )[["seed", "rmse", "deployable"]].rename(columns={
        "rmse": "current_rmse", "deployable": "current_deployable"
    })
    reference = salvage_seed.query(
        "family == @reference_family and policy == 'c0_only'"
    )[["seed", "rmse", "deployable"]].rename(columns={
        "rmse": "salvage_c0_rmse", "deployable": "salvage_deployable"
    })
    paired = current.merge(reference, on="seed", validate="one_to_one")
    assert sorted(paired["seed"].astype(int).tolist()) == sorted(seeds)
    paired["comparison"] = comparison_label
    paired["current_family"] = current_family
    paired["reference_family"] = reference_family
    paired["current_policy"] = "c0_only semantic dry oracle"
    paired["reference_policy"] = "c0_only historical salvage C0"
    paired["current_local_dry_expert_index"] = current_dry_index
    paired["reference_local_dry_expert_index"] = 0
    paired["delta_rmse_current_minus_salvage"] = (
        paired["current_rmse"] - paired["salvage_c0_rmse"]
    )
    paired["deployable"] = False
    paired["ece_used_for_comparison_or_fit"] = False
    paired_rows.append(paired)

paired_dry_comparison = pd.concat(paired_rows, ignore_index=True)
assert len(paired_dry_comparison) == 15
assert bool((paired_dry_comparison["current_deployable"] == False).all())
assert bool((paired_dry_comparison["salvage_deployable"] == False).all())
paired_dry_comparison.to_csv(ECE_DIR / "salvage_c0_dry_comparison_by_seed.csv", index=False)
dry_comparison_summary = paired_dry_comparison.groupby("comparison", as_index=False).agg(
    current_rmse_mean=("current_rmse", "mean"),
    current_rmse_sd=("current_rmse", "std"),
    salvage_c0_rmse_mean=("salvage_c0_rmse", "mean"),
    salvage_c0_rmse_sd=("salvage_c0_rmse", "std"),
    delta_rmse_mean=("delta_rmse_current_minus_salvage", "mean"),
    delta_rmse_sd=("delta_rmse_current_minus_salvage", "std"),
    n_seeds=("seed", "nunique"))
dry_comparison_summary.to_csv(ECE_DIR / "salvage_c0_dry_comparison_summary.csv", index=False)
print(dry_comparison_summary.to_string(index=False, float_format=lambda value: f"{value:.6f}"))

                                 comparison  current_rmse_mean  current_rmse_sd  salvage_c0_rmse_mean  salvage_c0_rmse_sd  delta_rmse_mean  delta_rmse_sd  n_seeds
             Backbone current vs salvage C0           0.057768         0.000617              0.057768            0.000617         0.000000       0.000000        5
Guarded local c1 dry vs salvage Backbone C0           0.192536         0.004021              0.057768            0.000617         0.134768       0.003603        5
                   V0 current vs salvage C0           0.057768         0.000617              0.057768            0.000617         0.000000       0.000000        5
